# Lab 9: Runtime & Retrieval — Execution Configuration + Retrieval-Augmented Agents

**Difficulty: Advanced | ~40 min | Requires Lab 8 (and Lab 5)**

Everything an agent can do at run time is decided by configuration you set *before* it runs. This lab turns two of those knobs. First, **how the agent executes**: LangGraph gives you a loop that can pause for approval before a tool fires and a hard bound on how many steps the loop may take — the difference between an agent that checks with you and an agent that retries a dead service forever. Second, **what the agent can access**: a retrieval-augmented agent pulls exactly the documents it needs out of a knowledge base on demand, instead of carrying the whole corpus in every prompt.

**Cost:** ~9 OpenRouter calls on the free model per full run. No other APIs, no servers to host, no network dependency beyond the model itself.

**Step 2 — Imports, the key, and the measuring instrument.**

This lab runs on the same free OpenRouter model as Labs 5–8, so the only per-run cost is model calls — no external servers, no API keys beyond the one in `.env`. `UsageCapture` is the same callback instrument from Lab 8: LangChain calls `on_llm_end` after every LLM call and the provider reports `prompt_tokens`, so you can watch the context budget move. The imports worth flagging are the LangGraph runtime pieces — `MemorySaver`, `Command`, and `GraphRecursionError` are the handles for the execution-config half of this lab.

In [ ]:
import os, re, math, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path(".env"))
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langgraph.errors import GraphRecursionError

def model():
    return ChatOpenAI(base_url="https://openrouter.ai/api/v1",
                      api_key=os.environ["OPENROUTER_API_KEY"],
                      model="nvidia/nemotron-3-super-120b-a12b:free", temperature=0)

class UsageCapture(BaseCallbackHandler):
    def __init__(self): self.calls = []
    def on_llm_end(self, response, **kwargs):
        usage = (response.llm_output or {}).get("token_usage", {})
        self.calls.append(usage.get("prompt_tokens", 0))

**Step 3 — The tools: one cooperative, one flaky.**

The runtime half of the lab uses two deliberately synthetic tools from the Meridian Trading world. `get_price` behaves itself — one call, one answer, ready to be paused and inspected. `run_etl` simulates the worst kind of production dependency: it fails with a transient `503` and its own description invites the caller to retry. The second tool exists to show what an agent does when nothing stops it — which is the entire argument for a runtime bound.

In [ ]:
@tool
def get_price(symbol: str) -> str:
    """Return the current synthetic price for a trading symbol."""
    prices = {"BTC": 61250, "ETH": 3390, "SOL": 142}
    p = prices.get(symbol.upper())
    return f"{symbol.upper()} is trading at ${p:,}" if p else f"No price for {symbol}"

@tool
def run_etl(job_id: str) -> str:
    """Submit an ETL job and report its status. The upstream warehouse is flaky:
    it returns a transient 503 error and expects callers to retry."""
    return "Error 503: warehouse is temporarily unavailable. Please retry the ETL job."

**Step 4 — Pause the loop: interrupt + checkpointer.**

By default an agent runs to completion and you only ever see the end. With a checkpointer and `interrupt_before=["tools"]`, the graph stops *between* the model deciding to call a tool and the tool actually firing. The run returns control to you mid-flight; `get_state()` shows which node is next and the pending tool call; and `Command(resume=...)` lets the loop continue. This is how human-in-the-loop approval flows work — a runtime configuration that decides *when* the agent is allowed to act.

In [ ]:
checkpointer = MemorySaver()
agent = create_agent(model=model(), tools=[get_price],
                     interrupt_before=["tools"], checkpointer=checkpointer)
cfg = {"configurable": {"thread_id": "runtime-demo"}}

agent.invoke({"messages": [("human", "What is the current BTC price?")]}, config=cfg)
state = agent.get_state(cfg)
print("paused before node:", state.next)
print("pending tool call :", state.values["messages"][-1].tool_calls)

agent.invoke(Command(resume="proceed"), config=cfg)
final = agent.get_state(cfg).values["messages"][-1].content
print("answer after resume:", str(final)[:100])

paused before node: ('tools',)
pending tool call : [{'name': 'get_price', 'args': {'symbol': 'BTC'}, 'id': 'chatcmpl-tool-ad9416712a49b71c', 'type': 'tool_call'}]


answer after resume: 

The current BTC price is $61,250.


**Step 5 — Bound the loop: recursion_limit.**

An agent's retry loop is unbounded by default — ask a model to get a job status from a service that keeps saying "try again", and it will keep trying. LangGraph's `recursion_limit` config is the runtime hard-stop. The system prompt below tells the agent that transient errors deserve a retry, so it has every excuse to keep calling `run_etl`; only the bound ends the run. (The limit counts graph nodes, not model calls — a limit of 8 is roughly four model-vs-tool cycles.)

In [ ]:
try:
    agent = create_agent(model=model(), tools=[run_etl],
                         system_prompt="You are a trading-ops automation agent. Transient errors (503) are "
                                      "expected from the warehouse — retry the ETL job until it succeeds.")
    agent.invoke({"messages": [("human", "Submit the ETL job j-1042 and report its status.")]},
                 config={"recursion_limit": 8})
    print("run completed normally")
except GraphRecursionError:
    print("GraphRecursionError: the loop hit the recursion_limit runtime bound")

GraphRecursionError: the loop hit the recursion_limit runtime bound


**Step 6 — The knowledge base.**

The retrieval half needs a body of facts the model does not already know. This is an internal wiki for the same Meridian Trading system — eight short documents covering risk limits, order types, rate limits, and the rest. None of this is in the model's training data as *your system's* policy, which is exactly why retrieval matters: the only way the agent can learn these rules is to go read them.

In [ ]:
DOCS = {
 "deploy-windows": "Production deploys happen on Tuesdays and Thursdays between 02:00 and 04:00 UTC. "
                    "A 15 minute freeze window blocks new orders while each deploy restarts the matching engine.",
 "risk-limits": "The risk engine enforces a maximum gross position of 50 BTC and 500 ETH. Per-order notional "
                 "is capped at 2,000,000 USD, and a kill switch halts all trading if realized daily loss "
                 "exceeds 500,000 USD.",
 "rate-limits": "The public API allows 120 requests per minute per API key. The websocket feed allows 20 "
                 "messages per second; bursts above 60 per minute disconnect the client for 60 seconds.",
 "incident-runbook": "On an exchange outage, stop placing new orders, keep existing positions open, and page "
                      "the on-call engineer through the #ops-major-incident channel. Do not manually close "
                      "positions during the first 30 minutes.",
 "model-config": "The inference model is nvidia/nemotron-3-super-120b-a12b:free served through OpenRouter at "
                  "temperature 0. It is retrained daily at 01:00 UTC, and a fallback chain swaps to a smaller "
                  "model after three consecutive 5xx errors.",
 "order-types": "Order types are limit, stop, and market. Market orders are only allowed for notional values "
                 "below 50,000 USD; anything larger must be placed as a limit order.",
 "settlement": "Perpetual funding is settled every 8 hours. Fees are 0.02 percent taker and 0.01 percent "
                "maker, and the minimum withdrawal is 0.001 BTC.",
 "support-escalation": "Support severity tiers are S1 through S3. S1 incidents page the on-call within 5 "
                        "minutes, S2 within 30 minutes, and S3 during business hours within 2 hours.",
}
corpus_text = "\n\n".join(f"[{t}] {d}" for t, d in DOCS.items())
print(f"{len(DOCS)} docs, {len(corpus_text)} chars in the corpus")

8 docs, 1567 chars in the corpus


**Step 7 — The retriever: BM25, implemented inline.**

Production retrieval is usually a vector store, but the scoring heart of a keyword retriever is one compact formula: BM25. It ranks a document by how many of the query's terms it contains, weighted by how rare each term is across the corpus (the IDF term) and length-normalized so long documents do not win by volume. We implement it inline — deterministic, no embeddings to download — and wrap it in a `kb_search` tool. The agent now has a way to pull knowledge into context *at runtime* instead of carrying the whole corpus with it.

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def bm25(query: str, docs: dict, k1: float = 1.5, b: float = 0.75) -> list[tuple[str, float]]:
    q_terms = tokenize(query)
    n = len(docs)
    avgdl = sum(len(tokenize(d)) for d in docs.values()) / n
    df = {t: sum(1 for d in docs.values() if t in tokenize(d)) for t in q_terms}
    scores = {}
    for title, text in docs.items():
        terms = tokenize(text); dl = len(terms)
        tf = {t: terms.count(t) for t in q_terms}
        s = 0.0
        for t in q_terms:
            idf = math.log(1 + (n - df.get(t, 0) + 0.5) / (df.get(t, 0) + 0.5))
            s += idf * tf.get(t, 0) * (k1 + 1) / (tf.get(t, 0) + k1 * (1 - b + b * dl / avgdl))
        scores[title] = s
    return sorted(scores.items(), key=lambda x: -x[1])

@tool
def kb_search(query: str) -> str:
    """Search the Meridian Trading knowledge base for the documents most relevant
    to a question and return them verbatim."""
    top = bm25(query, DOCS)[:2]
    return "\n\n".join(f"[{t}] {DOCS[t]}" for t, _ in top)

for title, score in bm25("Can I place a market order for $80,000 of BTC?", DOCS)[:2]:
    print(f"  top doc {title}: score {score:.2f}")

  top doc order-types: score 7.99
  top doc risk-limits: score 6.88


**Step 8 — Variant A: the whole corpus baked into the prompt.**

The naive approach: paste all eight documents into the system prompt and ask. It answers correctly — but `cap.calls[0]` is the decision-time context every request pays for, whether the question needs one document or all eight. This is the exact fixed-cost trap from Lab 8, now in the prompt itself.

In [ ]:
QUESTION = "Can I place a market order for $80,000 of BTC, and what is the largest position I may hold?"

baked = create_agent(model=model(),
                     system_prompt=f"You answer questions about the Meridian Trading system using ONLY "
                                  f"the internal knowledge base:\n\n{corpus_text}")
cap_baked = UsageCapture()
answer = baked.invoke({"messages": [("human", QUESTION)]}, config={"callbacks": [cap_baked]})
print("first-call input tokens:", cap_baked.calls[0])
print("answer:", str(answer["messages"][-1].content)[:120].replace("\n", " "))

first-call input tokens: 488
answer: No – market orders are only permitted for notional values under $50,000 USD. An $80,000 BTC order would exceed that limi


**Step 9 — Variant B: retrieval-augmented.**

Same question, same answer — but the agent starts with a small system prompt and a single tool. It calls `kb_search` at runtime, pays for the two relevant documents only on the *second* call, and answers from them. Compare the first-call numbers with Step 8: the retrieval agent's decision-time context is smaller, and the gap only widens as the corpus grows.

In [ ]:
retrieval = create_agent(model=model(), tools=[kb_search],
                         system_prompt="You are a trading-ops assistant. Use the kb_search tool to find "
                                      "facts about the Meridian Trading system before answering.")
cap_retrieval = UsageCapture()
answer = retrieval.invoke({"messages": [("human", QUESTION)]},
                          config={"callbacks": [cap_retrieval]})
print("per-call input tokens:", cap_retrieval.calls)
print("answer:", str(answer["messages"][-1].content)[:120].replace("\n", " "))

per-call input tokens: [336, 494]
answer:   No, you cannot place a market order for $80,000 of BTC. According to the Meridian Trading system’s order‑type rules, m


**Step 10 — Close the loop.**

Side by side, the two strategies tell the Lab 8 story from the other direction. Baked-in context is a fixed cost paid on every request regardless of need; retrieval turns it into a variable cost, paid only when the agent decides it needs the knowledge. The runtime half played the same trick from the other side of the equation — interrupts and recursion limits decide *when* the loop may act and *when* it must stop.

In [ ]:
print("decision-time context (first LLM call):")
print(f"  baked-in : {cap_baked.calls[0]:>6} tokens  (whole corpus in the system prompt)")
print(f"  retrieval: {cap_retrieval.calls[0]:>6} tokens  (only the tool schema)")
print(f"  retrieval second call: {cap_retrieval.calls[1]:>6} tokens  (pays for the 2 docs it retrieved)")

decision-time context (first LLM call):
  baked-in :    488 tokens  (whole corpus in the system prompt)
  retrieval:    336 tokens  (only the tool schema)
  retrieval second call:    494 tokens  (pays for the 2 docs it retrieved)
